ACT: 动作分块变换器 (Action Chunking with Transformers), VLA中的重要baseline

ACT的核心想法是一次预测一段未来的动作序列，而不是只预测一步

ACT核心技术

1.动作分块

将连续的动作序列分成固定长度为K的“块”，一次预测一个块

输入：当前观测

输出：动作序列 a = [at, at+1, ......, at+k-1]

2.时间集成

仅采用动作分块的话，在t时刻进行预测，执行动作序列到t+k-1, 后续再预测会导致轨迹不连续

采用重叠预测＋指数平均的方法解决

In [ ]:
时间步 t=0:   预测 [a_0, a_1, ..., a_99]
时间步 t=1:   预测 [a_1', a_2', ..., a_100']
时间步 t=2:   预测 [a_2'', a_3'', ..., a_101'']
...

执行 a_t 时，融合所有对 a_t 的预测:
a_t^{final} = Σ w_i * a_t^{(i)}

指数加权公式：

$ w_i = \exp(-m \cdot i) $

其中 m 是衰减系数 (通常 m=0.01)，$i$ 是预测的"年龄"(越新的预测权重越大)。

3.CVAE架构

训练时：

编码器：将观测o和真实动作a编码为隐变量z的分布

解码器：从隐变量z和观测o中重建动作序列

损失：1.重建损失：重建的动作序列和真实动作序列做损失

2.KL散度约束隐变量z，让他接近正态分布，方便后续推理采样

推理时：

从标准正态分布采样z ~ N(0，I)

解码器根据z和观测o生成动作序列

In [ ]:
import torch
import torch.nn as nn
from torch.nn import TransformerEncoder, TransformerDecoder

class ACTPolicy(nn.Module):
    def __init__(self, obs_dim, action_dim, chunk_size, z_dim, hidden_dim, num_encoder_layers, num_decoder_layers):
        super(ACTPolicy, self).__init__()
        self.chunk_size = chunk_size
        self.z_dim = z_dim

        self.obs_encoder = nn.Linear(obs_dim, hidden_dim)

        self.cvae_encoder = TransformerEncoder(
            nn.TransformerEncoderLayer(d_model = hidden_dim, nhead = 8),
            num_layers = num_encoder_layers 
        )

        self.z_mu = nn.Linear(hidden_dim, z_dim)
        self.z_logvar = nn.Linear(hidden_dim. z_dim)

        self.action_queries = nn.Parameter(torch.randn(chunk_size, hidden_dim))
        self.z_proj = nn.Linear(z_dim, hidden_dim)
        self.action_decoder = TransformerDecoder(
            nn.TransformerDecoderLayer(d_model=hidden_dim, nhead=8),
            num_layers=num_decoder_layers
        )
        self.action_head = nn.Linear(hidden_dim, action_dim)

        self.action_embed = nn.Linear(action_dim, hidden_dim)
    
    '''CVAE编码器'''
    def encode(self, obs_feat, actions):
        actions_feat = self.action_embed(actions)

        combined = torch.cat([obs_feat, actions_feat], dim = 1)
        combined = combined.permute(1, 0, 2)

        encoded = self.cvae_encoder(combined)
        pooled = encoded.mean(dim = 0)

        return self.z_mu(pooled), self.z_logvar(pooled)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    '''CVAE解码器'''
    def decode(self, obs_feat, z):
        batch_size = obs_feat.shape[0]

        queries = self.action_queries.unsqueeze(0).expand(batch_size, -1, -1)
        z_expanded = self.z_proj(z).unsqueeze(1)
        queries = queries + z_expanded

        queries = queries.permute(1, 0, 2)
        obs_feat = obs_feat.permute(1, 0, 2)

        decoded = self.action_decoder(queries, obs_feat)
        decoded = decoded.permute(1, 0, 2)
        
        return self.action_head(decoded)
    
    def forward(self, obs, actions = None):
        obs_feat = self.obs_encoder(obs).unsqueeze(1)

        if actions is not None:
            z_mu, z_logvar = self.encode(obs_feat, actions)
            z = self.reparameterize(z_mu, z_logvar)
            pred_actions = self.decode(obs_feat, z)
            return pred_actions, z_mu, z_logvar
        else:
            z = torch.randn(obs.shape[0], self.z_dim, device=obs.device)
            pred_actions = self.decode(obs_feat, z)
            return pred_actions

def compute_loss(pred_actions, gt_actions, z_mu, z_logvar, beta=10.0):
    """ACT 损失函数: 重建损失 + β * KL 散度"""
    recon_loss = nn.functional.mse_loss(pred_actions, gt_actions)
    kl_loss = -0.5 * torch.mean(1 + z_logvar - z_mu.pow(2) - z_logvar.exp())
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

'''时间集成'''
class TemporalEnsemble:
    def __init__(self, chunk_size, action_dim, decay=0.01):
        self.chunk_size = chunk_size
        self.decay = decay
        self.action_buffer = []  # 存储历史预测
        self.weights = []        # 存储对应权重
    
    def update(self, new_chunk):
        """添加新预测的 chunk"""
        # new_chunk: [k, action_dim]
        self.action_buffer.append(new_chunk)
        self.weights.append(1.0)
        
        # 衰减旧权重
        self.weights = [w * (1 - self.decay) for w in self.weights]
        
        # 移除过期的预测
        while len(self.action_buffer) > self.chunk_size:
            self.action_buffer.pop(0)
            self.weights.pop(0)
    
    def get_action(self, t):
        """获取时刻 t 的集成动作"""
        weighted_sum = 0
        total_weight = 0
        
        for i, (chunk, w) in enumerate(zip(self.action_buffer, self.weights)):
            # 计算该 chunk 对时刻 t 的预测
            chunk_start_time = t - len(self.action_buffer) + i + 1
            local_idx = t - chunk_start_time
            
            if 0 <= local_idx < len(chunk):
                weighted_sum += w * chunk[local_idx]
                total_weight += w
        
        return weighted_sum / total_weight if total_weight > 0 else None
